In [0]:
## Spark Performance Settings
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
# dbutils.widgets.text(name="start_date", defaultValue="2026-05-07", label="start_date")
# dbutils.widgets.text(name="lookback_period", defaultValue="30", label="lookback_period")
# dbutils.widgets.text(name="year_lookback_period", defaultValue="365", label="year_lookback_period")
# dbutils.widgets.text(name="catalog_schema_prefix", defaultValue="marketingdata_dev.claire_wilsonbarnes", label="catalog_schema_prefix")
# dbutils.widgets.text(name="training_run", defaultValue="True", label="training_run")
# dbutils.widgets.text(name="earliest_date", defaultValue="2025-12-01", label="earliest_date")
# dbutils.widgets.text(name="latest_date", defaultValue="2026-05-27", label="latest_date")

In [0]:
%sql

CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix|| '.pctr_views_aggregated') AS (
SELECT 
      pv.account_number 
    , c.rundate
    , COALESCE(SUM(pv.viewtimespentsecs) FILTER (WHERE DAYOFWEEK(pv.timestamp)= dayofweek(c.rundate - interval '1 day'))/ SUM(pv.viewtimespentsecs), 0) AS perc_viewtimedow
    , COALESCE(COUNT(Distinct pv.department),0) AS number_departments_viewed 
    , COALESCE(COUNT(pv.account_number),0) AS number_pages_viewed 
    , COALESCE(COUNT(pv.account_number)  FILTER (WHERE  pv.timestamp BETWEEN c.rundate - interval '8 days' AND c.rundate - interval '1 day'),0 ) AS number_pages_viewed_last_week
FROM 
    IDENTIFIER(:catalog_schema_prefix || '.pctr_training_customer_base') AS c 
    LEFT JOIN IDENTIFIER(:catalog_schema_prefix|| '.pctr_build_page_views') AS pv
        ON pv.account_number=c.account_number
        AND pv.viewdate BETWEEN c.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND c.rundate - INTERVAL '1' DAY

GROUP BY   
    pv.account_number 
    , c.rundate
);


In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog_schema_prefix|| '.pctr_view_themes') AS (
WITH cte_latest_views AS (
  SELECT 
    pv.* 
  , c.rundate
  , SUM(pv.viewtimespentsecs) OVER (PARTITION BY pv.account_number) AS total_time_spent 
FROM 
  IDENTIFIER(:catalog_schema_prefix|| '.pctr_build_page_views')  AS pv
  INNER JOIN IDENTIFIER(:catalog_schema_prefix || '.pctr_training_customer_base') AS c 
    ON pv.account_number=c.account_number
    AND pv.viewdate BETWEEN c.rundate - (INTERVAL '1 DAY' * (:lookback_period + 1)) AND c.rundate - INTERVAL '1' DAY
-- WHERE 
--    --taking last 50 items viewed
--     pv.view_order <=50 
)
SELECT 
  pv.account_number
, pv.rundate
,  regexp_replace(t.theme, '[^a-zA-Z0-9]', '') AS themes
 -- Without date/time accounted for 
-- , SUM(((pv.viewtimespentsecs / pv.total_time_spent)* 1/t.theme_rank::numeric) / (date_diff(pv.rundate, pv.timestamp::date) +1)) AS view_theme_score
, SUM(((pv.viewtimespentsecs / pv.total_time_spent)* 1/t.theme_rank::numeric)  * (exp(-0.0231 * (datediff(pv.rundate, pv.timestamp::date))))) AS view_theme_score
FROM 
    cte_latest_views AS pv
    INNER JOIN marketingdata_prod.warehouse.next_uk_nextads_item_themes AS t 
        ON pv.pid=t.pid
        -- want current day themes
        AND t.rundate=pv.rundate
GROUP BY 
  pv.account_number
, pv.rundate
, themes
); 
